# Compiled kernels: cold and warm Darcy timings

PyMHM compiles selected array operations in its numerical owners. The user's
workflow stays **meshes → local equations → global equations → assemble → solve
→ fields and errors**. We write the equations here instead of calling a prepared
Darcy solver. NumPy arrays and Python providers remain the public interface.

This small notebook needs only `pymhm`, NumPy, Matplotlib and threadpoolctl. It
runs a fresh 2D control, visualizes its actual macro mesh, and repeats the same
formulation in 3D. Its in-process timings are illustrative: the first call may
load an existing Numba cache. The separately archived paired campaign measures
fresh-process imports, empty-cache compilation and complete warmed solves on
identical discretizations. A numerical-kernel speedup is distinct from a
complete simulation speedup.

In [ ]:
from functools import partial
from time import perf_counter

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
from threadpoolctl import threadpool_limits

from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble
from pymhm.execution.cpu import ExecutionConfig
from pymhm.fem.scalar.operators import boundary_data
from pymhm.fem.scalar.tetrahedron import tetra_operators
from pymhm.fem.scalar.triangle import scalar_operators, trace_coupling
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.fem.traces.triangle_3d import (
    TriangularSkeleton,
    tetra_boundary_data,
    tetra_trace_coupling,
)
from pymhm.meshes.tetrahedron import TetraMesh
from pymhm.meshes.triangle import TriangleMesh
from pymhm.postprocessing.nodal import nodal_field

# Optional notebook preparation/execution tools.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/a6207a09d3833b07ba127a77a26d0edc74de8a658c62d8ab5ab52e349182b336/numba_kernel_performance-companion.zip"
COMPANION_SHA256 = "a6207a09d3833b07ba127a77a26d0edc74de8a658c62d8ab5ab52e349182b336"
# This notebook itself imports only installed packages.

## 1. Write the physical problem and independent exact data

On the unit square or cube, solve

$$
\begin{aligned}
q&=-K\nabla p, & \nabla\cdot q&=f, & p\vert_{\partial\Omega}&=0,\\
K(x)&=\exp\left(\alpha\prod_{i=1}^d\sin(2\pi x_i/\varepsilon)\right),
& p_\star(x)&=\prod_{i=1}^d\sin(\pi x_i),\\
f&=d\pi^2Kp_\star-\nabla K\cdot\nabla p_\star.
\end{aligned}
$$

Set $\alpha=0.25$ and $\varepsilon=0.137$. The nonaligned period gives different
local material matrices; no material-equivalence reuse is assumed. The source
uses analytical derivatives independently of the discrete diffusion operator.

In [ ]:
PERIOD, AMPLITUDE = 0.137, 0.25


def exact_pressure(points):
    """Evaluate the manufactured pressure at physical coordinates."""
    return np.prod(np.sin(np.pi * points), axis=-1)


def exact_gradient(points):
    """Differentiate the product without division at its zero factors."""
    dimension = points.shape[-1]
    sine, cosine = np.sin(np.pi * points), np.cos(np.pi * points)
    return np.pi * np.stack(
        [
            cosine[..., i] * np.prod(sine[..., [j for j in range(dimension) if j != i]], axis=-1)
            for i in range(dimension)
        ],
        axis=-1,
    )


def permeability(points):
    """Evaluate positive scalar isotropic permeability."""
    return np.exp(AMPLITUDE * np.prod(np.sin(2 * np.pi * points / PERIOD), axis=-1))


def source(points):
    """Apply minus divergence using independent exact material derivatives."""
    dimension, omega = points.shape[-1], 2 * np.pi / PERIOD
    sine, cosine = np.sin(omega * points), np.cos(omega * points)
    material = permeability(points)
    grad_material = (
        omega
        * AMPLITUDE
        * material[..., None]
        * np.stack(
            [
                cosine[..., i]
                * np.prod(sine[..., [j for j in range(dimension) if j != i]], axis=-1)
                for i in range(dimension)
            ],
            axis=-1,
        )
    )
    return dimension * np.pi**2 * material * exact_pressure(points) - np.sum(
        grad_material * exact_gradient(points), axis=-1
    )

## 2. Define macro and local meshes before the equations

The 2D macro mesh has 32 triangles. Every macro edge is divided into eight
local intervals. Continuous $P_2$ pressure lives on each local mesh; a continuous piecewise linear
normal-flux multiplier lives on two segments of each macroface. Each macroelement has one
retained constant pressure coordinate. The global skeleton normal orientation
is owned by the mesh/trace objects.
This field illustration enriches the trace to resolve the reconstructed fields.
The archived optimization comparison keeps its separately declared P0 trace
spaces fixed in both implementations; its accuracy and timing scope remain
explicit.


In [ ]:
DEGREE, LOCAL_REFINEMENT, ORDER = 2, 8, 6
macro = TriangleMesh.unit_square(4)
skeleton = SkeletonSpace(
    macro, tuple(FaceSpace.uniform(1, 2, continuous=True) for _ in macro.faces)
)
local_meshes = tuple(macro.submesh(cell, LOCAL_REFINEMENT) for cell in range(len(macro.cells)))
print("Macro triangles:", len(macro.cells))
print("Total local triangles:", sum(len(mesh.cells) for mesh in local_meshes))
print("Skeleton unknowns:", skeleton.size)

## 3. Translate the local variational equation

For each macroelement $T$, the local equation and global pairing are

$$
\begin{aligned}
(K\nabla p_T,\nabla v)_T+\langle\lambda,v\rangle_{\partial T}&=(f,v)_T,\\
-\sum_T\langle\mu,p_T\rangle_{\partial T}&=0.
\end{aligned}
$$

Thus `a` is the diffusion Gram matrix, `L` the source load, `b` the oriented
trace pairing and `c=-b.T` the declared global pairing. The constant local
kernel is explicit, and `mass @ constant` specifies its physical-volume moment.
The FEM functions below are assembly conveniences: they implement the displayed
integrals using Basix tabulation and the shared numerical owners. Numba stays
inside these owners; providers and mathematical objects stay in Python.

In [ ]:
def local_problem(cell, *, mesh, trace, fine_meshes):
    """Declare local and global forms without choosing a physical solver."""
    fine = fine_meshes[cell]
    if fine.points.shape[1] == 2:
        a, mass, L = scalar_operators(
            fine, DEGREE, diffusion=permeability, source=source, order=ORDER
        )
        b = trace_coupling(mesh, cell, fine, trace, DEGREE)
    else:
        a, mass, L = tetra_operators(
            fine, DEGREE, diffusion=permeability, source=source, order=ORDER
        )
        b = tetra_trace_coupling(mesh, cell, fine, trace, DEGREE)
    constant = np.ones((len(L), 1))
    return LocalEquations(
        a=a,
        L=L,
        b=b,
        c=-b.T,
        dofs=trace.cell_dofs(cell),
        kernel=constant,
        moments=mass @ constant,
        metadata=fine,
        field_data=(nodal_field("pressure", fine, DEGREE),),
    )

## 4. Declare the global problem, assemble, then solve

Homogeneous weak pressure data give zero exterior load. `Equation` carries
that global load, followed by one retained coordinate per macroelement.
`assemble` builds and eliminates local problems and sums shared face entries
in a fixed order; `solve` solves the reduced global system and reconstructs
local pressure. The kernel's Numba leaves are serial, with BLAS limited to one
thread, so this cell does not create nested thread pools.

In [ ]:
boundary, fixed = boundary_data(skeleton, 0.0, order=ORDER)
problem = MultiscaleProblem(
    global_equation=Equation(0, -np.r_[boundary, np.zeros(len(macro.cells))]),
    local_provider=partial(local_problem, mesh=macro, trace=skeleton, fine_meshes=local_meshes),
    items=range(len(macro.cells)),
    trace_size=skeleton.size,
    coarse_sizes=(1,) * len(macro.cells),
    fixed=fixed,
)

with threadpool_limits(limits=1):
    start = perf_counter()
    system = assemble(problem, execution=ExecutionConfig(backend="serial", workers=1))
    result = system.solve()
    first_seconds = perf_counter() - start

print("First in-process assembly/solve:", first_seconds, "s")
print("Global compatibility residual:", result.residual)
assert result.residual <= 1e-10

In [ ]:
original_row_errors = []
for response, values in zip(system.responses, result.fields, strict=True):
    local = response.problem
    trace = result.trace[local.trace_dofs]
    defect = local.matrix @ values + local.coupling @ trace - local.load
    scale = abs(local.matrix) @ abs(values) + abs(local.coupling) @ abs(trace) + abs(local.load)
    original_row_errors.append(
        np.linalg.norm(defect) / max(np.linalg.norm(scale), np.finfo(float).tiny)
    )
print("Original pressure-equation L2 backward error:", max(original_row_errors))
assert max(original_row_errors) <= 1e-10

## 5. Plot independent pressure, physical flux and error values

Named fields retain the executed basis and one-sided macroelement coefficients.
Each fine triangle is sampled through its own owner; reconstructed interfaces
are not averaged. The raw physical flux is $-K\nabla p_h$; macro conservation
belongs to the normal-flux skeleton, not fine-cell conservation of this raw
primal gradient. Every panel displays the actual macro mesh, including the
analytical and error panels, and owns its own color scale.

In [ ]:
point_blocks, value_blocks, gradient_blocks, triangle_blocks = [], [], [], []
offset = 0
for field in result.field("pressure"):
    fine = field.mesh
    points = fine.points[fine.cells].reshape(-1, 2)
    owners = np.repeat(np.arange(len(fine.cells)), 3)
    point_blocks.append(points)
    value_blocks.append(field.evaluate(points, cells=owners))
    gradient_blocks.append(field.gradient(points, cells=owners))
    triangle_blocks.append(np.arange(len(points)).reshape(-1, 3) + offset)
    offset += len(points)
points = np.concatenate(point_blocks)
pressure = np.concatenate(value_blocks)
gradient = np.concatenate(gradient_blocks)
triangles = np.concatenate(triangle_blocks)
flux = -permeability(points)[:, None] * gradient
triangulation = mtri.Triangulation(points[:, 0], points[:, 1], triangles)

panels = (
    ("Permeability K", permeability(points)),
    ("Analytical pressure", exact_pressure(points)),
    ("MHM pressure", pressure),
    ("Pressure error magnitude", abs(pressure - exact_pressure(points))),
    ("Physical Darcy flux magnitude", np.linalg.norm(flux, axis=1)),
    (
        "Physical flux error magnitude",
        np.linalg.norm(flux + permeability(points)[:, None] * exact_gradient(points), axis=1),
    ),
)
fig, axes = plt.subplots(2, 3, figsize=(11.8, 7.5), constrained_layout=True)
for axis, (title, values) in zip(axes.flat, panels, strict=True):
    image = axis.tripcolor(triangulation, values, shading="gouraud", cmap="viridis")
    for face in macro.faces:
        endpoints = macro.points[face]
        axis.plot(endpoints[:, 0], endpoints[:, 1], color="black", linewidth=0.45, alpha=0.65)
    axis.set(title=title, xlabel="x", ylabel="y", aspect="equal")
    fig.colorbar(image, ax=axis, shrink=0.83, pad=0.025)
plt.show()

## 6. Separate first-call costs from warmed repeated assembly

Each repetition assembles new local matrices and factors and a new reduced
system. It reuses only the already declared geometry/data and compiled code.
These small notebook timers include assembly, the reduced solve and pressure
reconstruction. They exclude geometry description, plotting and norm
integration; the complete archived campaign includes those numerical setup and
physical post-processing costs. These scopes are intentionally labeled rather
than mixed in a single speedup.

In [ ]:
warm_seconds = []
with threadpool_limits(limits=1):
    for _ in range(3):
        start = perf_counter()
        repeated = assemble(problem, execution=ExecutionConfig(backend="serial", workers=1))
        repeated_result = repeated.solve()
        warm_seconds.append(perf_counter() - start)
        np.testing.assert_allclose(repeated_result.trace, result.trace, rtol=1e-10, atol=1e-11)
print("Warm repetitions:", warm_seconds)
print("Warm median:", float(np.median(warm_seconds)), "s")

## 7. Use the same abstractions in three dimensions

Tetrahedral local spaces use the same variational forms and constant physical
kernel, with triangular normal-flux macrofaces. Degree and quadrature remain
explicit. This small solve verifies that the compiled scalar contraction accepts
three coordinate components; it is not a large-problem scalability study.

In [ ]:
macro3d = TetraMesh.unit_cube(1)
skeleton3d = TriangularSkeleton(macro3d, 2, degree=1)
local_meshes3d = tuple(macro3d.submesh(cell, 4) for cell in range(len(macro3d.cells)))
boundary3d, fixed3d = tetra_boundary_data(skeleton3d, 0.0, {}, ORDER)
problem3d = MultiscaleProblem(
    global_equation=Equation(0, -np.r_[boundary3d, np.zeros(len(macro3d.cells))]),
    local_provider=partial(
        local_problem, mesh=macro3d, trace=skeleton3d, fine_meshes=local_meshes3d
    ),
    items=range(len(macro3d.cells)),
    trace_size=skeleton3d.size,
    coarse_sizes=(1,) * len(macro3d.cells),
    fixed=fixed3d,
)
with threadpool_limits(limits=1):
    system3d = assemble(problem3d, execution=ExecutionConfig(backend="serial", workers=1))
    result3d = system3d.solve()
print("3D trace DOFs:", skeleton3d.size)
print("3D global compatibility residual:", result3d.residual)
assert result3d.residual <= 1e-10

In [ ]:
from itertools import combinations

fig3d = plt.figure(figsize=(11.8, 4.8), layout="constrained")
axes3d = [fig3d.add_subplot(1, 2, i + 1, projection="3d") for i in range(2)]
centers3d, values3d = [], []
for field in result3d.field("pressure"):
    centers = field.mesh.points[field.mesh.cells].mean(axis=1)
    centers3d.append(centers)
    values3d.append(field.evaluate(centers, cells=np.arange(len(field.mesh.cells))))
centers3d, pressure3d = np.concatenate(centers3d), np.concatenate(values3d)
macro_edges3d = sorted(
    {tuple(sorted(edge)) for tetra in macro3d.cells for edge in combinations(tetra, 2)}
)
for axis, title, values in zip(
    axes3d,
    ("3D analytical pressure", "3D MHM pressure"),
    (exact_pressure(centers3d), pressure3d),
    strict=True,
):
    image = axis.scatter(*centers3d.T, c=values, cmap="viridis", s=22, depthshade=False)
    for edge in macro_edges3d:
        endpoints = macro3d.points[list(edge)]
        axis.plot(*endpoints.T, color="black", linewidth=0.55, alpha=0.45)
    axis.set(title=title, xlabel="x", ylabel="y", zlabel="z")
    axis.set_box_aspect((1, 1, 1))
    fig3d.colorbar(image, ax=axis, shrink=0.74, pad=0.08)
plt.show()

## 8. Interpret the measured optimization campaign

Read the [performance report](https://ipes-lncc.github.io/pymhm/performance/#compiled-array-kernels)
for the paired 2D/3D timings, physical pressure/flux errors, original-equation
controls and separate kernel measurements. The campaign archives every sample,
the actual basis/coefficient vectors, complete source identities, package
versions, CPU affinity and native thread limits. Empty per-process Numba caches
separate compilation from warmed execution. All local operators and factors are
rebuilt; no special material-equivalence optimization is used.

The importable acquisition helper is
[`examples/numba_performance.py`](https://github.com/ipes-lncc/pymhm/blob/main/examples/numba_performance.py).
It owns timing/reporting and uses the same package numerical owners. It is a
checkout/companion helper, not part of the installed `pymhm` API. Numerical
algorithms stay in the library. Its `acquire` function accepts the stated 2D/3D
mesh/space cases and writes a new report; `compare_reports` verifies matched
coordinates and physical cases before calculating speedups.

Numba compiles selected scalar integration, ordered sparse reduction and planar
sampling kernels. Basix tabulation, UFL/FEniCSx adapters and external LU/AMG
solvers keep their native implementations. Compensated binary64 accumulation
retains the physical controls without automatically promoting these hot
integrations to `longdouble`. Explicit extended-precision solver/refinement
options remain separate capabilities. This CPU campaign does not establish a
GPU gain or multi-node scalability.